# LLM finetuning

Steps in this notebook: 
1. Import libraries required for finetuning
2. Set variables
3. Define preprocessing function (tokenization)
4. Load tokenizer and model
5. Apply LoRA
6. Load and set up the data
7. Set up the HuggingFace Trainer
8. Start the finetuning process
9. Monitor GPU utilization from terminal while the training is running
10. Save the model
11. BONUS: Check SLURM job ID
12. Restart kernel

# 1. Import libraries required for finetuning

Before we can train a model, we need to load the tools we'll use. Think of these as the specialized software packages that handle different parts of the process:

- **torch** — PyTorch, the core deep learning framework that runs computations on the GPU
- **transformers** — Hugging Face's library for loading and working with pretrained language models
- **peft** — Enables *LoRA*, a technique that makes finetuning much cheaper by only training a small fraction of the model's parameters
- **datasets** — Handles loading and processing our training data efficiently
- **pandas** — General-purpose data manipulation, used here to load our JSON data file

These are already installed to the chosen module.


In [1]:
import argparse
import os
import sys
import time
import torch
#import mlflow
import pandas as pd

from datasets import Dataset 

from pathlib import Path
from datasets import load_from_disk
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    AutoProcessor
)
from functools import partial

import warnings
warnings.filterwarnings("ignore")

## 2. Set variables that will be used e.g. in model loading, finetuning and loading data

- **`SLURM_JOB_ACCOUNT` and `SLURM_JOB_USER`** — read automatically from the LUMI environment, used to build file paths
- **`input_model`** — the pretrained model we start from: [Qwen/Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). Changing this to another Hugging Face model ID will finetune that model instead
- **`json_file`** — path to your training data file
- **`batch_size`** — how many examples the model processes at once. Higher is faster but uses more GPU memory — `2` is a safe starting point for a single GPU
- **`max_tokens`** — maximum length of a single training example in tokens (roughly words). Examples longer than this are cut off at the end

In [2]:
SLURM_JOB_ACCOUNT = os.getenv("SLURM_JOB_ACCOUNT")
USER = os.getenv("SLURM_JOB_USER")

print(f"Billing project:  {SLURM_JOB_ACCOUNT}")
print(f"Username:         {USER}")

Billing project:  project_462000131
Username:         hintsala


In [3]:
input_model = "Qwen/Qwen3-4B-Instruct-2507"
output_path = f"/scratch/{SLURM_JOB_ACCOUNT}/{USER}/health_case/ft_model"
model_output_name = f"{input_model}_finetuned"
#model_output_name = f"{input_model.split('/')[-1]}_finetuned"
json_file = f"/scratch/{SLURM_JOB_ACCOUNT}/data/structured_notes.json" # vaihda pienempään?
batch_size = 2
cache_dir = f"/scratch/{SLURM_JOB_ACCOUNT}/hf-cache/hub"
max_tokens = 2048
num_workers = int(os.getenv("SLURM_CPUS_PER_TASK"))

output_model_dir = os.path.join(output_path, model_output_name)
merged_output_dir = os.path.join(
    output_path,
    f"{model_output_name}_merged"
)

print(f"Merged model output dir: {merged_output_dir}")

Merged model output dir: /scratch/project_462000131/hintsala/health_case/ft_model/Qwen/Qwen3-4B-Instruct-2507_finetuned_merged


## 3. Define preprocessing function (tokenization)

This cell defines the `system_prompt` and a `preprocess` function that converts raw conversations into tokenized format the model can learn from.

Models don't read text directly — they read **tokens**, numbers representing words or parts of words. The function builds each training example as a three-part conversation, tokenizes it, then masks the prompt portion in the labels with `-100` so the model is only trained to predict the assistant's response, not repeat the question back.

The three conversation roles are:
- `system` — instructions that tell the model what its job is (here: converting doctor-patient dialogue to a clinical note)
- `user` — the raw doctor-patient conversation (the input)
- `assistant` — the correct structured note we want the model to produce (the expected output)

In [4]:
system_prompt = """You are a medical clinical documentation assistant. 
You task is to convert a dialogue between a doctor and patient into a structured clinical note in the following output format:
REASON FOR VISIT:
<Brief summary of why the patient is seeking care>
PATIENT DETAILS AND HISTORY:
<Age, gender, relevant demographics, relevant past medical history, conditions, medications, surgeries, lifestyle factors>
CURRENT STATUS:
<Current symptoms, findings, vitals, clinical observations>
TREATMENTS/ACTIONS:
<Medications prescribed, procedures performed, advice given>
FOLLOW-UP PLAN:
<Next steps, monitoring, referrals, timelines. Follow-up plan should not include "future" details that are mentioned in the note, but rather should infer what the next steps would be based on the found future details.>
"""

def preprocess(examples, tokenizer, system_prompt, max_tokens=2048):
    """Convert input_data/output pairs into tokenized chat format."""
    input_ids_list = []
    labels_list = []
    attention_mask_list = []

    for input_data, output in zip(examples["conversation"], examples["structured_note"]):

        # Build chat messages
        messages = [
            {
                "role": "system",
                "content": system_prompt},
            {"role": "user", "content": input_data},
            {"role": "assistant", "content": output},
        ]

        # Apply chat template — tokenize full conversation
        full_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        # Also build prompt-only part to know where assistant response starts
        prompt_messages = messages[:-1]
        prompt_text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        # Tokenize full conversation
        tokenized = tokenizer(
            full_text,
            truncation=True,
            max_length=max_tokens,
            padding=False,
        )

        # Tokenize prompt only to get its length
        prompt_tokenized = tokenizer(
            prompt_text,
            truncation=True,
            max_length=max_tokens,
            padding=False,
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        prompt_len = len(prompt_tokenized["input_ids"])

        # Mask prompt tokens in labels — only compute loss on assistant response
        labels = [-100] * prompt_len + input_ids[prompt_len:]

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list,
    }

print(f"Used system_prompt:\n\n{system_prompt}")

Used system_prompt:

You are a medical clinical documentation assistant. 
You task is to convert a dialogue between a doctor and patient into a structured clinical note in the following output format:
REASON FOR VISIT:
<Brief summary of why the patient is seeking care>
PATIENT DETAILS AND HISTORY:
<Age, gender, relevant demographics, relevant past medical history, conditions, medications, surgeries, lifestyle factors>
CURRENT STATUS:
<Current symptoms, findings, vitals, clinical observations>
TREATMENTS/ACTIONS:
<Medications prescribed, procedures performed, advice given>
FOLLOW-UP PLAN:
<Next steps, monitoring, referrals, timelines. Follow-up plan should not include "future" details that are mentioned in the note, but rather should infer what the next steps would be based on the found future details.>



## 4. Load tokenizer & models

These cells detect the GPU, load the pretrained model and tokenizer, then wrap the model with LoRA.

**GPU detection** — checks whether a GPU is available. Training on CPU would be impractically slow — a step that takes 1 second on GPU can take 50–100 seconds on CPU.

**Tokenizer** — must match the model exactly as each model has its own vocabulary. The `pad_token` fix ensures the tokenizer can handle batches of different-length examples.

In [5]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
tokenizer = AutoTokenizer.from_pretrained(input_model, use_fast=True, cache_dir=cache_dir)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded | Vocab size: {len(tokenizer)} | Pad token: {tokenizer.pad_token}")
print(f"GPU memory before model load: {torch.cuda.memory_allocated() / 1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB total")

Tokenizer loaded | Vocab size: 151669 | Pad token: <|endoftext|>
GPU memory before model load: 0.00 GB / 68.70 GB total


In [7]:
model = AutoModelForCausalLM.from_pretrained(
    input_model,
    torch_dtype=torch.bfloat16,
    device_map=device,
    cache_dir=cache_dir
)

print(f"GPU memory after model load: {torch.cuda.memory_allocated() / 1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB total")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

GPU memory after model load: 8.04 GB / 68.70 GB total


## 5. Apply LoRA

**LoRA (Low-Rank Adaptation)** — freezes the original model weights and adds small trainable adapter layers on top, making finetuning feasible on a single GPU. 

Key settings:
- `r=16` — size of the adapters, higher means more capacity but more memory
- `lora_alpha=8` — scaling factor for adapter outputs
- `lora_dropout=0.05` — randomly disables 5% of adapter connections to prevent overfitting
- `target_modules="all-linear"` — applies LoRA to all linear layers

`model.print_trainable_parameters()` will confirm only ~0.5–1% of parameters are being trained.



## 5. Apply LoRA
The amount of trainable parameters depends on the model architecture, the `r` value, and which layers LoRA is applied to. With our configuration on this 4B model, only ~0.89% of parameters are trained (~38M out of 4.3B total).

Key settings:
- `r=16` — the rank, controls the size of the adapters. Higher means more learning capacity but more memory. Common values are 8, 16, and 32
- `lora_alpha=8` — a scaling factor for the adapter outputs, typically set to `r/2` or equal to `r`
- `lora_dropout=0.05` — randomly disables 5% of adapter connections during training to prevent the model from memorising the training data
- `target_modules="all-linear"` — applies LoRA to all linear layers in the model. You could instead target only specific layers such as attention layers, which would reduce the trainable parameter count further

In [8]:
peft_config = LoraConfig(
    lora_alpha=8,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

In [9]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


# Load and setup the data

This cell loads the training data from disk and splits it into two subsets: one for training and one for validation.

### Where does the data come from?
The data is loaded from the `raw_val` folder that was saved by the SLURM training script. This is the pre-saved subset of your full dataset, stored in Hugging Face's efficient Arrow format via `save_to_disk()`.

### Why split into train and validation?
- **Training set** — the model learns from these examples
- **Validation set** — held back during training and used to measure how well the model generalises to data it hasn't seen. If training loss keeps dropping but validation loss stops improving or gets worse, the model is overfitting (memorising rather than learning)

The `seed=42` ensures the split is reproducible — running this again will always produce the same train/val split, which is important for fair comparisons between experiments.

### ⚠️ Note on the data source
This notebook loads from the pre-saved `raw_val` folder rather than the full training JSON. This means it is training and validating on a subset of the original validation split only, not the full dataset. If you want to train on the full data, change the loading line to:
```python
df = pd.read_json(json_file)
dataset = Dataset.from_pandas(df, preserve_index=False)
In this notebook, we use the validation split (3k conversations) of the bigger dataset (30k conversations). This validation dataset will be used to conduct a smaller scale finetuning suitable in this notebook environment.

## 6. Load and set up the data

This cell loads the training data and splits it into two subsets: one for training and one for validation.

The full dataset contains 30 000 doctor-patient conversation and structured note pairs, but here we use only 2 000 examples — enough for a meaningful finetuning run within the time and memory limits of a single-GPU notebook session.

The dataset has several columns but we only use two:
- **`conversation`** — the raw doctor-patient dialogue, used as the model input
- **`structured_note`** — the target clinical note the model learns to produce

The data is split 95/5 into training and validation sets. The validation set is held back during training to measure how well the model generalises to unseen examples. `seed=42` ensures the split is the same every time you run the cell, which is important for reproducibility.

In [10]:
## DATA
df = pd.read_json(json_file)[:2000]
dataset = Dataset.from_pandas(df, preserve_index=False)
split = dataset.train_test_split(test_size=0.05, seed=42)

raw_train = split["train"]
raw_val = split["test"]

print(f"  Train size: {len(raw_train)}")
print(f"  Val size:   {len(raw_val)}")


  Train size: 1900
  Val size:   100


We will also quickly inspect at some of the examples of the data. `conversation` and `structured_notes` will be used as the finetuning material.

In [11]:
dataset_df = pd.DataFrame(raw_val)

In [12]:
df.head(1)

,idx,note,full_note,conversation,summary,text_for_llm,structured_note
0,155216,"A a sixteen year-old girl, presented to our Ou...","A a sixteen year-old girl, presented to our Ou...","Doctor: Good morning, what brings you to the O...","{\n""visit motivation"": ""Discomfort in the neck...","[{'role': 'system', 'content': [{'type': 'text...",**REASON FOR VISIT:** \nThe patient presents ...


In [13]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.html.use_mathjax', False)
df[['conversation', 'structured_note']].head(1).style.set_properties(**{'white-space': 'pre-wrap', 'width': '500px'})

,conversation,structured_note
0,"Doctor: Good morning, what brings you to the Outpatient department today? Patient: Good morning doctor, I have some discomfort in my neck and lower back, and I'm not able to maintain an erect posture. Doctor: Hmm, okay. Can you tell me more about the discomfort? Patient: Yes, I tend to fall on either side when I stand up from a sitting position, and my head is always turned to the right and upwards. Doctor: I see. Are you experiencing any pain in your neck? Patient: Yes, I have pain and discomfort in my neck. Doctor: Okay. And what about your back? Patient: There is a sideways bending in my lumbar region. To counter the abnormal positioning of my back and neck, I have to keep my limbs in a specific position to allow my body weight to be supported. Doctor: I understand. Does this restriction of body movements affect your daily life? Patient: Yes, I need assistance in standing and walking, and my parents have to help me with my daily chores, including all activities of self-care. Doctor: I see. How long have you been experiencing these difficulties? Patient: I've been experiencing these difficulties for the past four months since I was introduced to olanzapine tablets for the control of my exacerbated mental illness. Doctor: I see. And you've been diagnosed with bipolar affective disorder, correct? Patient: Yes, I was diagnosed with bipolar affective disorder seven years ago. Doctor: And you've been taking olanzapine for your mental illness for seven years, correct? Patient: Yes, I have. My first episode of the affective disorder was mania when I was eleven, and I've been taking olanzapine tablets in 2.5-10 mg doses per day at different times. Doctor: I see. So, you developed pain and discomfort in your neck within the second week of being put on olanzapine at a dose of 5 mg per day, correct? Patient: Yes, that's correct. The sustained and abnormal contraction of my neck muscles pulls my head to the right in an upward direction. Doctor: I see. And these features have persisted for the first three years of your illness with a varying intensity, distress, and dysfunction, correct? Patient: Yes, that's correct. The intensity, distress, and dysfunction tend to correlate with the dose of olanzapine. Doctor: I see. And apart from a brief period of around three weeks when you were given trihexyphenidyl 4 mg per day for rigidity in your upper limbs, you were not prescribed any other psychotropic medication, correct? Patient: Yes, that's correct. The rigidity showed good response to trihexyphenidyl 4 mg per day. Doctor: Okay. I'm going to order some tests for you, and I'll be able to give you a proper diagnosis after that. Patient: Okay, doctor. Doctor: I'll also instruct you on follow-up requirements. Patient: Okay, thank you, doctor.","**REASON FOR VISIT:** The patient presents with persistent neck and lower‑back dystonia causing painful posturing, restriction of movement, and dependence on assistance for standing and daily activities. **PATIENT DETAILS AND HISTORY:** A 16‑year‑old female with a seven‑year history of bipolar affective disorder, first manifesting as mania at age eleven. She has been treated intermittently with olanzapine (2.5 – 10 mg daily), sodium valproate 1000 mg daily, lithium carbonate 450 mg daily, escitalopram for depressive features, and briefly with trihexyphenidyl and propranolol for rigidity. Long‑term use of olanzapine has been associated with the onset of dystonic symptoms. She also has hypothyroidism secondary to lithium, for which she receives levothyroxine (Eltroxin) 50 µg daily. No surgical history; lifestyle factors are not reported. There is no family history of dystonia. **CURRENT STATUS:** Since re‑introduction of olanzapine at 10 mg daily, the patient has experienced sustained neck muscle contraction pulling the head rightward and upward, lateral flexion of the lumbar spine, and inability to maintain an erect posture, requiring support for standing and walking. Pain

In [14]:
for i, row in df[["conversation", "structured_note"]].head(1).iterrows():
    print(f"{'='*80}")
    print(f"EXAMPLE {i+1}")
    print(f"\n--- CONVERSATION ---\n{row['conversation']}")
    print(f"\n--- STRUCTURED NOTE ---\n{row['structured_note']}")
    print(f"{'='*80}\n")

EXAMPLE 1

--- CONVERSATION ---
Doctor: Good morning, what brings you to the Outpatient department today?
Patient: Good morning doctor, I have some discomfort in my neck and lower back, and I'm not able to maintain an erect posture.
Doctor: Hmm, okay. Can you tell me more about the discomfort?
Patient: Yes, I tend to fall on either side when I stand up from a sitting position, and my head is always turned to the right and upwards.
Doctor: I see. Are you experiencing any pain in your neck?
Patient: Yes, I have pain and discomfort in my neck.
Doctor: Okay. And what about your back?
Patient: There is a sideways bending in my lumbar region. To counter the abnormal positioning of my back and neck, I have to keep my limbs in a specific position to allow my body weight to be supported.
Doctor: I understand. Does this restriction of body movements affect your daily life?
Patient: Yes, I need assistance in standing and walking, and my parents have to help me with my daily chores, including all 

## Tokenize datasets

In [15]:
preprocess_fn = partial(preprocess, system_prompt=system_prompt, tokenizer=tokenizer, max_tokens=max_tokens)

In [16]:
tokenized_train = raw_train.map(
    preprocess_fn,
    batched=True,
    remove_columns=raw_train.column_names,
    num_proc=num_workers,
)

Map (num_proc=7):   0%|          | 0/1900 [00:00<?, ? examples/s]

In [17]:
tokenized_val = raw_val.map(
    preprocess_fn,
    batched=True,
    remove_columns=raw_val.column_names,
    num_proc=num_workers,
)

Map (num_proc=7):   0%|          | 0/100 [00:00<?, ? examples/s]

# DEELETE this part

These cells set up everything the training loop needs before it can start: the training behaviour, the data batching strategy, and the Trainer object that ties it all together.

### TrainingArguments — controlling how training behaves
This is where all the training knobs live. The most important ones:

- **`learning_rate=2e-5`** — how large a step the optimizer takes when updating the LoRA weights each iteration. Too high and training becomes unstable, too low and it learns very slowly. `2e-5` is a safe, commonly used value for finetuning
- **`num_train_epochs=1`** — how many times the model sees the full training dataset. One epoch is often enough for finetuning on a task-specific dataset, especially with LoRA
- **`per_device_train_batch_size`** — how many examples are processed at once during training. Set in the config cell
- **`per_device_eval_batch_size=batch_size*8`** — validation can use a much larger batch since no gradients are stored, so we use 8x the training batch size to speed it up
- **`eval_steps=150`** — run validation every 150 training steps to track how well the model generalises
- **`save_steps=300`** — save a checkpoint every 300 steps. Checkpoints let you recover if the session ends unexpectedly
- **`save_total_limit=3`** — keep only the 3 most recent checkpoints to avoid filling up disk space
- **`load_best_model_at_end=True`** — after training finishes, automatically restore the checkpoint that had the lowest validation loss rather than using the final one
- **`weight_decay=0.01`** — a small regularization penalty that discourages the model from over-relying on any single parameter, helping prevent overfitting
- **`bf16=True`** — use bfloat16 precision to halve memory usage with minimal impact on quality
- **`save_safetensors=True`** — save checkpoints in the safer `.safetensors` format rather than the older pickle-based `.bin` format
- **`report_to=["mlflow"]`** — send loss and metrics to MLflow every `logging_steps` so you can track the run visually

### DataCollatorForSeq2Seq — batching examples together
Individual training examples have different lengths. The collator pads them to the same length so they can be processed as a batch. `pad_to_multiple_of=8` aligns lengths to multiples of 8, which is more efficient on GPU hardware.

### Trainer — the training orchestrator
The `Trainer` object combines everything: the model, the arguments, the datasets, the tokenizer, and the collator. Calling `trainer.train()` in the next cell is all that's needed to start the full training loop.

## 7. Setting up the HuggingFace Trainer with required arguments for finetuning

These cells configure the training behaviour, set up data batching, and initialize the Trainer that orchestrates everything.

**`TrainingArguments`** defines how training behaves: learning rate, how many epochs to train, how often to evaluate and save checkpoints, and what precision to use. The most important ones are set in the config cell — the rest are sensible defaults for LoRA finetuning.

**`DataCollatorForSeq2Seq`** pads examples to the same length so they can be processed as a batch, aligned to multiples of 8 for GPU efficiency.

**`Trainer`** ties everything together — model, arguments, datasets, tokenizer, and collator. The next cell starts training with a single call to `trainer.train()`.

In [18]:
training_args = TrainingArguments(
    disable_tqdm=False,
    output_dir=output_model_dir,
    save_strategy="steps",
    save_steps=300,
    save_total_limit=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    bf16=True,
    load_best_model_at_end=True,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size*8,
    dataloader_num_workers=num_workers,
    ddp_find_unused_parameters=True,
    dataloader_pin_memory=True,
    save_safetensors=True,
    metric_for_best_model="eval_loss",
    eval_strategy="steps",
    eval_steps=150,
    num_train_epochs=1,
    report_to=["mlflow"],
    logging_steps=50,
    logging_strategy="steps",
    run_name=f"{model_output_name}_{os.environ.get('SLURM_JOB_ID', 'local')}",
)

print(f"Training for {training_args.num_train_epochs} epoch(s) | Eval every {training_args.eval_steps} steps | Save every {training_args.save_steps} steps | LR: {training_args.learning_rate}")

Training for 1 epoch(s) | Eval every 150 steps | Save every 300 steps | LR: 2e-05


In [19]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)

print("Data collator ready")

Data collator ready


In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print(f"Trainer ready | Train examples: {len(tokenized_train)} | Val examples: {len(tokenized_val)}")

Trainer ready | Train examples: 1900 | Val examples: 100


# 8. Start the finetuning process

This cell starts the training. During training you will see a table printed every 150 steps showing:

- **Step** — how many training batches have been processed so far
- **Training Loss** — how wrong the model's predictions are on the training data, should decrease over time
- **Validation Loss** — how wrong the predictions are on the held-out validation examples the model has never seen. If this stops improving while training loss keeps dropping, the model is starting to memorise the training data rather than learning general patterns

With 2000 examples and `batch_size=2` on a single GPU expect roughly 20 minutes.

> **Note:** You may see an MLflow message appearing during training — this is expected and can be ignored. `report_to=["mlflow"]` is set in the `TrainigArguments` purely to prevent an automatic connection to an external logging service (Weights & Biases) that would otherwise cause errors in this environment.

In [ ]:
start_train = time.time()

trainer.train()

stop_train = time.time()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
150,1.240300,1.238660
300,1.201200,1.161558
450,1.107400,1.130545
600,1.109100,1.114641


In [ ]:
elapsed = stop_train - start_train
hours   = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)
print(f"Training took: {hours}h {minutes}m {seconds}s")

## 9. Monitor GPU utilization from terminals while the training is running

While training is running, open a **new terminal inside the Jupyter App** and run:

```bash
hostname    # confirm which compute node you are on
rocm-smi    # show current GPU memory usage and utilization
```

`rocm-smi` is the AMD equivalent of `nvidia-smi` — it shows GPU memory usage, temperature, and compute utilization. Run it manually a few times during training to see the GPU being utilized. You should see one GPU active with memory usage climbing once training starts.

If you submitted via `sbatch` instead, monitoring works differently — you would need to find the job ID and open an interactive session on the compute node manually. See the [LUMI AI Guide](https://github.com/Lumi-supercomputer/LUMI-AI-Guide/tree/main/01-quickstart#readme) for step-by-step instructions.

## 10. Merge LoRA adapters and save the model

This is the final step — merging the trained LoRA adapters back into the base model and saving everything to disk.

During training, LoRA kept the original model weights frozen and only updated the small adapter layers. `merge_and_unload()` combines these adapters back into the base model weights, producing a single standalone model that can be loaded and run without any dependency on the PEFT library.

The merged model, tokenizer, and processor are all saved to `merged_output_dir` defined in the config cell. This folder is used as input in the next inference notebook.

In [ ]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained(
    merged_output_dir,
    safe_serialization=False
)

tokenizer.save_pretrained(merged_output_dir)
processor.save_pretrained(merged_output_dir)

## 11. Check SLURM Job ID

In a typical HPC workflow you submit a job from the terminal with `sbatch my_script.sh` and — if there are no errors in the script setup — get a job ID back immediately. The job then runs unattended in the background.

This notebook works differently. When you launched the Jupyter App on LUMI and selected your resources (GPUs, memory, time limit), the system automatically submitted a SLURM job behind the scenes to reserve those resources. **The Jupyter session itself is the SLURM job** — all code you run in this notebook executes within that already-allocated job.

This means:
- The cell below prints the job ID of your current Jupyter session
- When the time limit expires, the session ends and any running cells are interrupted
- There is no separate SLURM script needed for this notebook — unlike the multi-GPU `finetune.py` which is submitted via `sbatch` and runs without a Jupyter session

In [ ]:
# Check Slurm job ID
print(os.environ.get("SLURM_JOB_ID", "Not running inside a SLURM job"))

# 12. Restart kernel before continuing

Before moving to the inference notebook, restart the kernel to free the GPU memory occupied by the finetuned model and trainer. Without this, the next notebook may run out of memory when trying to load a model.

**Kernel → Restart Kernel** in the Jupyter menu, or run the cell below.

In [ ]:
from IPython import get_ipython

get_ipython().kernel.do_shutdown(restart=True)

## Move on to next notebook

[./inference_nb.ipynb](./inference_nb.ipynb)